In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
import torch
print("Visible GPU:", torch.cuda.get_device_name(0))
print("Memory free:", torch.cuda.mem_get_info()[0] / 1e9, "GB")

Visible GPU: Quadro RTX 5000
Memory free: 15.780216832 GB


## Step 1 - Environment setup & `iter_passages()` loader

`iter_passages()` is the **single source of truth** for the entire RAG pipeline.  It iterates the FEVER Wikipedia dump (`wiki-pages/*.jsonl`) and yields one tuple per sentence: `(passage_id, page_id, sentence_idx, text)`.  Both the BM25 index and the Qdrant vector index are built by consuming this same generator in the same order, which guarantees that BM25 document IDs, Qdrant point IDs, and `(page_id, sentence_idx)` keys are perfectly aligned across the two indexes.

**Why sentence-level, not page-level?**  FEVER gold evidence annotations record the exact `(page_id, sentence_idx)` pair.  Indexing at sentence granularity means retrieval quality can be evaluated for free — checking whether gold evidence appears in the top-10 results requires no extra tooling.  Page-level retrieval would require a second-pass extraction and would silently lose any gold sentence ranked outside the top-K pages (a hard recall ceiling with no easy fix).

**Downstream dependencies.**  Step 2 (BM25 build) and Step 4 (Qdrant embed + upsert) both consume this generator.  The `passage_id` string `"{page_id}::{sentence_idx}"` is used as the stable key in both indexes and as the lookup key when the verifier needs to fetch evidence text.

| Choice | Why |
|---|---|
| Sentence granularity | Aligns with FEVER annotation units; enables free retrieval-quality eval |
| `{page_id}::{sentence_idx}` stable key | Deterministic; survives re-runs without index rebuild |
| Generator, not list | 25 M sentences would OOM if fully materialised; streaming keeps RAM flat |
| Skip empty / whitespace entries | FEVER `lines` field contains blank entries between section headers |
| Reuse `clean_artifacts` | Same bracket encoding (`-LRB-`, `-RRB-`, etc.) appears in wiki dump as in training data |

In [3]:
%pip install qdrant-client         -q
%pip install rank_bm25             -q
%pip install sentence-transformers -q
%pip install tqdm                  -q
%pip install jsonlines             -q



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install --upgrade Pillow

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
# Force user-installed packages (e.g. Pillow 12.x, torch 2.2.1+cu121) before system packages.
_user_site = "/home/ai-ws2/.local/lib/python3.10/site-packages"
if _user_site not in sys.path:
    sys.path.insert(1, _user_site)

import os
import re
import glob
import json
import pickle
import random
import itertools
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

# ── Reproducibility + device ──────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Import transformers while torch is confirmed available ────────────────
# Must happen here so transformers caches is_torch_available()=True before
# any other cell imports from it.
import transformers
from transformers import AutoModel
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")

# ── Knowledge base ────────────────────────────────────────────────────────
WIKI_DIR        = "Data_wiki/Wikipedia dump/wiki-pages"
PAGE_INDEX_FILE = "page_to_file_index.json"

# ── Output paths (relative to notebook dir) ──────────────────────────────
QDRANT_PATH  = "rag/indices/qdrant"
BM25_PATH    = "rag/indices/bm25/bm25_okapi.pkl"
PASSAGE_MAP  = "rag/indices/bm25/passage_ids.pkl"

# ── Embedding model ───────────────────────────────────────────────────────
EMBED_MODEL    = "jinaai/jina-embeddings-v5-text-small"
EMBED_DIM      = 1024  # confirmed: jina-embeddings-v5-text-small native dim
EMBED_BATCH    = 64
EMBED_MAX_LEN  = 256
EMBED_DTYPE    = torch.float16

# ── Qdrant ────────────────────────────────────────────────────────────────
COLLECTION        = "fever_wiki"
HNSW_M            = 16
HNSW_EF_CONSTRUCT = 200

# ── Smoke test ────────────────────────────────────────────────────────────
SMOKE_LIMIT = 10_000

os.makedirs(QDRANT_PATH, exist_ok=True)
print(f"WIKI_DIR    : {WIKI_DIR}")
print(f"SMOKE_LIMIT : {SMOKE_LIMIT:,}")

Device : cuda
GPU    : Quadro RTX 5000
VRAM   : 16.7 GB
torch        : 2.2.1+cu121
transformers : 4.57.6
WIKI_DIR    : Data_wiki/Wikipedia dump/wiki-pages
SMOKE_LIMIT : 10,000


In [6]:
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.device_count())


True
12.1
1


In [15]:
import subprocess
result = subprocess.run(
    ["pip", "install", "torch==2.2.1+cu121", "torchvision", "torchaudio",
     "--index-url", "https://download.pytorch.org/whl/cu121", "-q"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.8.0 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip



In [4]:
# clean_artifacts is defined earlier in GP.ipynb (the NEI-pairs section).
# If starting a fresh kernel here, copy that cell in or run it first.

def clean_artifacts(text):
    text = text.replace("-LRB-", "(").replace("-RRB-", ")")
    text = text.replace("-LSB-", "[").replace("-RSB-", "]")
    text = text.replace("-LCB-", "{").replace("-RCB-", "}")
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def iter_passages(wiki_dir, limit=None):
    """Yield (passage_id, page_id, sentence_idx, text) from FEVER wiki-pages dump."""
    n = 0
    for fpath in sorted(glob.glob(os.path.join(wiki_dir, "*.jsonl"))):
        with open(fpath, encoding="utf-8") as fh:
            for line in fh:
                doc       = json.loads(line)
                page_id   = doc["id"]
                raw_lines = doc.get("lines", "")
                for raw in raw_lines.split("\n"):
                    raw = raw.strip()
                    if not raw:
                        continue
                    parts = raw.split("\t")
                    if len(parts) < 2:
                        continue
                    idx_str = parts[0]
                    if not idx_str.isdigit():
                        continue
                    text = parts[1].strip()  # parts[2:] are link annotations — discard
                    if not text:
                        continue
                    text         = clean_artifacts(text)
                    sentence_idx = int(idx_str)
                    passage_id   = f"{page_id}::{sentence_idx}"
                    yield passage_id, page_id, sentence_idx, text
                    n += 1
                    if limit is not None and n >= limit:
                        return

print("iter_passages() defined.")

iter_passages() defined.


In [10]:
first_file = sorted(glob.glob(os.path.join(WIKI_DIR, "*.jsonl")))[0]
print(f"Smoke-check file : {os.path.basename(first_file)}\n")

# ── First 5 passages ──────────────────────────────────────────────────────
print("Sample passages:")
for pid, page, sidx, text in itertools.islice(iter_passages(WIKI_DIR), 5):
    print(f"  passage_id   : {pid}")
    print(f"  page_id      : {page}")
    print(f"  sentence_idx : {sidx}")
    print(f"  text         : {text[:100]}")
    print()

# ── Passage count in first file only ──────────────────────────────────────────
count = 0
with open(first_file, encoding="utf-8") as fh:
    for line in fh:
        doc = json.loads(line)
        for raw in doc.get("lines", "").split("\n"):
            raw = raw.strip()
            if not raw:
                continue
            parts = raw.split("\t", 1)
            if len(parts) == 2 and parts[1].strip():
                count += 1

print(f"Passages in {os.path.basename(first_file)} : {count:,}")

Smoke-check file : wiki-001.jsonl

Sample passages:
  passage_id   : 1928_in_association_football::0
  page_id      : 1928_in_association_football
  sentence_idx : 0
  text         : The following are the football ( soccer ) events of the year 1928 throughout the world .

  passage_id   : 1986_NBA_Finals::0
  page_id      : 1986_NBA_Finals
  sentence_idx : 0
  text         : The 1986 NBA Finals was the championship round of the 1985 -- 86 NBA season . NBA National Basketbal

  passage_id   : 1986_NBA_Finals::1
  page_id      : 1986_NBA_Finals
  sentence_idx : 1
  text         : It pitted the Eastern Conference champion Boston Celtics against the Western Conference champion Hou

  passage_id   : 1986_NBA_Finals::2
  page_id      : 1986_NBA_Finals
  sentence_idx : 2
  text         : The Celtics defeated the Rockets four games to two to win their 16th NBA championship . NBA National

  passage_id   : 1986_NBA_Finals::3
  page_id      : 1986_NBA_Finals
  sentence_idx : 3
  text         : T

## Step 2 - BM25 index build

BM25 (Okapi variant via `rank_bm25`) is the lexical leg of the hybrid retrieval pipeline. It scores passages by term frequency normalised for document length and weighted by corpus-level IDF — making it strong at matching rare proper nouns, which dominate FEVER claims. Building BM25 first (CPU, no GPU required) also validates the passage loader end-to-end before committing GPU hours to the Qdrant embedding pass.

**Tokenisation.** Lowercase + `re.findall(r'\w+', text)` — no stemming, no explicit stopword removal. IDF down-weights high-frequency terms automatically, and stemming would degrade proper-noun matching (entity names like "Einstein" or "Meryl Streep" survive `\w+` cleanly; a Porter stemmer would mangle them). This matches the tokenisation used in GP.ipynb's NEI mining step.

**Downstream dependencies.** The `BM25Okapi` object and its parallel `passage_ids` list are saved to `rag/indices/bm25/` as pickles. Position `i` in `passage_ids` always corresponds to document `i` inside the BM25 object — `bm25.get_scores(q)[i]` ↔ `passage_ids[i]`. Step 5 (sanity check) and the Day 2 hybrid retrieval cell both load from this path. The smoke index (`*_smoke.pkl`) covers only `wiki-001.jsonl` and is used here to validate pipeline correctness before the full 25 M-sentence build.

| Choice | Why |
|---|---|
| `BM25Okapi` | Same library used in NEI mining (GP.ipynb); no new dependency |
| Lowercase + `\w+` split | Simple, fast; no stemming to corrupt entity names |
| No stopword removal | IDF handles weighting naturally; explicit removal risks clipping meaningful short terms |
| Parallel `passage_ids` list | Position-aligned with BM25 internals; O(1) id lookup at query time |
| `rag/indices/bm25/` output | Isolated from other index artifacts; 

In [17]:
from rank_bm25 import BM25Okapi

BM25_SMOKE_PATH = "rag/indices/bm25/bm25_smoke.pkl"
BM25_SMOKE_IDS  = "rag/indices/bm25/passage_ids_smoke.pkl"
os.makedirs("rag/indices/bm25", exist_ok=True)

def tokenize(text):
    return re.findall(r'\w+', text.lower())

# ── Read first wiki file only ─────────────────────────────────────────────
SMOKE_FILE = sorted(glob.glob(os.path.join(WIKI_DIR, "*.jsonl")))[0]
print(f"Source file      : {os.path.basename(SMOKE_FILE)}\n")

smoke_ids    = []
smoke_tokens = []
smoke_texts  = []

with open(SMOKE_FILE, encoding="utf-8") as fh:
    for raw_line in tqdm(fh, desc="Tokenising", unit=" docs"):
        doc     = json.loads(raw_line)
        page_id = doc["id"]
        for raw in doc.get("lines", "").split("\n"):
            raw = raw.strip()
            if not raw:
                continue
            parts = raw.split("\t")
            if len(parts) < 2:
                continue
            idx_str = parts[0]
            if not idx_str.isdigit():
                continue
            text = parts[1].strip()  # parts[2:] are link annotations — discard
            if not text:
                continue
            text = clean_artifacts(text)
            smoke_ids.append(f"{page_id}::{int(idx_str)}")
            smoke_tokens.append(tokenize(text))
            smoke_texts.append(text)

print(f"Passages indexed : {len(smoke_ids):,}")
print(f"Building BM25Okapi  ...")
bm25_smoke = BM25Okapi(smoke_tokens)

with open(BM25_SMOKE_PATH, "wb") as f:
    pickle.dump(bm25_smoke, f)
with open(BM25_SMOKE_IDS, "wb") as f:
    pickle.dump(smoke_ids, f)

print(f"Saved BM25 object : {BM25_SMOKE_PATH}")
print(f"Saved passage IDs : {BM25_SMOKE_IDS}")

Source file      : wiki-001.jsonl



Tokenising: 0 docs [00:00, ? docs/s]

Passages indexed : 170,546
Building BM25Okapi  ...
Saved BM25 object : rag/indices/bm25/bm25_smoke.pkl
Saved passage IDs : rag/indices/bm25/passage_ids_smoke.pkl


In [18]:
# ── Sanity query ──────────────────────────────────────────────────────────
QUERY    = "Albert Einstein Nobel Prize"
q_tokens = tokenize(QUERY)
scores   = bm25_smoke.get_scores(q_tokens)
top_idxs = scores.argsort()[::-1][:5]

print(f"Query : \"{QUERY}\"\n")
print(f"  {'Rank':<4} {'Score':>8}  passage_id")
print(f"  {'':4} {'':8}  text")
print("  " + "─" * 76)
for rank, idx in enumerate(top_idxs, 1):
    print(f"  {rank:<4}  {scores[idx]:>8.3f}  {smoke_ids[idx]}")
    print(f"            {smoke_texts[idx][:110]}")
    print()

# Alignment check: tokens[i] must match tokenize(texts[i]) for the top hit
best = top_idxs[0]
print(f"Alignment check — tokens[{best}][:5]         : {smoke_tokens[best][:5]}")
print(f"                  tokenize(texts[{best}])[:5] : {tokenize(smoke_texts[best])[:5]}")

Query : "Albert Einstein Nobel Prize"

  Rank    Score  passage_id
                 text
  ────────────────────────────────────────────────────────────────────────────
  1       20.059  1,2-Wittig_rearrangement::1
            The reaction is named for Nobel Prize winning chemist Georg Wittig .

  2       18.870  '47_-LRB-magazine-RRB-::20
            Nathaniel Benchley ventured `` Up in Benchley 's Room '' and Albert Einstein recommended a few science books .

  3       16.204  1GOAL_Education_for_All::6
            Alongside Queen Rania , it is co-chaired by FIFA President Sepp Blatter and Nobel prize laureate Archbishop De

  4       14.990  -LRB-137924-RRB-_2000_BD19::13
            is considered a good candidate for measuring the effects of Albert Einstein 's general theory of relativity be

  5       14.163  1979_Sakharov::22
            This minor planet was named in honour of renowned Russian mathematician and physicist Andrei Sakharov ( 1921 -

Alignment check — tokens[43329][:

### Step 2A - Full-corpus dry-pass count

A count-only pass over the complete FEVER wiki dump before committing to the full build. `iter_passages()` is called end-to-end but every tuple is discarded immediately — no tokenisation, no accumulation. This does two things: (1) confirms the expected ~25 M sentence count, and (2) exercises every file in the dump, surfacing any malformed JSON or encoding errors before the 45+ minute build begins. If this cell crashes, fix the underlying file issue before running 2B.

The elapsed time also gives a baseline for raw I/O speed — if the dry pass takes more than ~5 minutes, the full build will be I/O-bound rather than CPU-bound. The count is written to `corpus_count.txt` so Cell 2B can use it for a meaningful `tqdm` ETA without re-running this cell.

In [19]:
import time

COUNT_FILE = "rag/indices/bm25/corpus_count.txt"
os.makedirs("rag/indices/bm25", exist_ok=True)

print("Dry-pass: iterating full corpus (count only, no tokenisation) ...")
t0 = time.time()

total_passages = sum(1 for _ in iter_passages(WIKI_DIR))

elapsed = time.time() - t0
print(f"Total passages : {total_passages:,}")
print(f"Elapsed        : {elapsed:.1f} s  ({elapsed / 60:.1f} min)")

with open(COUNT_FILE, "w") as f:
    f.write(str(total_passages))
print(f"Count saved to : {COUNT_FILE}")

Dry-pass: iterating full corpus (count only, no tokenisation) ...
Total passages : 25,247,890
Elapsed        : 306.9 s  (5.1 min)
Count saved to : rag/indices/bm25/corpus_count.txt


### Step 2B - Full BM25 build

Second pass over the full corpus: tokenise every passage and accumulate two parallel lists — `full_ids` (passage ID strings) and `token_lists` (tokenised texts) — then construct a single `BM25Okapi` object in one shot. The build must be single-shot because `rank_bm25` computes IDF statistics over the entire corpus at construction time; incremental updates are not supported.

`tokenize()` is the same lowercase + `\w+` regex function used in the smoke cell. If `corpus_count.txt` exists from Cell 2A, `tqdm` receives an accurate `total=` for a meaningful ETA; otherwise it falls back to a count-only bar with a warning. Memory note: at ~25 M passages, `token_lists` can reach 10–15 GB of Python objects — well within the lab machine's 188 GB, but visible in `htop` during the build.

Both the BM25 object and the passage ID list are saved to disk at the end of this cell. If the kernel dies before saving, the full build must be re-run from scratch.

In [20]:
import time

try:
    import psutil
    _has_psutil = True
except ImportError:
    _has_psutil = False

BM25_FULL_PATH   = "rag/indices/bm25/bm25_okapi.pkl"
PASSAGE_IDS_PATH = "rag/indices/bm25/passage_ids.pkl"
COUNT_FILE       = "rag/indices/bm25/corpus_count.txt"
os.makedirs("rag/indices/bm25", exist_ok=True)

def tokenize(text):
    return re.findall(r'\w+', text.lower())

# ── Resolve corpus count for tqdm ETA ────────────────────────────────────
if os.path.exists(COUNT_FILE):
    with open(COUNT_FILE) as f:
        _total = int(f.read().strip())
    print(f"Corpus count (from 2A) : {_total:,}")
else:
    _total = None
    print("WARNING: corpus_count.txt not found — run Cell 2A first for ETA. Proceeding without total.")

# ── Tokenise + accumulate ─────────────────────────────────────────────────
full_ids    = []
token_lists = []

t0 = time.time()
for pid, _, _, text in tqdm(iter_passages(WIKI_DIR), total=_total,
                             desc="Tokenising", unit=" passages"):
    full_ids.append(pid)
    token_lists.append(tokenize(text))

elapsed_tok = time.time() - t0
print(f"\nPassages collected : {len(full_ids):,}")
print(f"Tokenisation time  : {elapsed_tok:.1f} s  ({elapsed_tok / 60:.1f} min)")

# ── Build BM25Okapi ───────────────────────────────────────────────────────
print("Building BM25Okapi (single-shot IDF computation) ...")
t1 = time.time()
bm25_full = BM25Okapi(token_lists)
elapsed_bm25 = time.time() - t1
print(f"BM25 build time    : {elapsed_bm25:.1f} s  ({elapsed_bm25 / 60:.1f} min)")

if _has_psutil:
    rss_gb = psutil.Process().memory_info().rss / 1e9
    print(f"RSS memory (approx): {rss_gb:.1f} GB")

# ── Save ──────────────────────────────────────────────────────────────────
print(f"\nSaving ...")
t2 = time.time()
with open(BM25_FULL_PATH, "wb") as f:
    pickle.dump(bm25_full, f)
with open(PASSAGE_IDS_PATH, "wb") as f:
    pickle.dump(full_ids, f)
elapsed_save = time.time() - t2
print(f"Saved BM25 object  : {BM25_FULL_PATH}")
print(f"Saved passage IDs  : {PASSAGE_IDS_PATH}")
print(f"Save time          : {elapsed_save:.1f} s")

Corpus count (from 2A) : 25,247,890


Tokenising:   0%|          | 0/25247890 [00:00<?, ? passages/s]


Passages collected : 25,247,890
Tokenisation time  : 741.5 s  (12.4 min)
Building BM25Okapi (single-shot IDF computation) ...
BM25 build time    : 337.2 s  (5.6 min)
RSS memory (approx): 55.5 GB

Saving ...
Saved BM25 object  : rag/indices/bm25/bm25_okapi.pkl
Saved passage IDs  : rag/indices/bm25/passage_ids.pkl
Save time          : 316.1 s


### Step 2C - Persistence & load helper

Defines `load_bm25_index()` — the standard entry point for all downstream cells that need the full BM25 index. Loading deserialises the pickled `BM25Okapi` object and its parallel `passage_ids` list from disk, restoring them to the exact state from Cell 2B. Any cell in Day 2 or later that needs to issue a BM25 query calls this function rather than rebuilding.

The sanity check below runs the same "Albert Einstein Nobel Prize" query used on the smoke index. With the full 25 M-sentence corpus, the top `passage_id` values should now include entries whose `page_id` component is `Albert_Einstein` — confirming the full index was built and loaded correctly. Passage text is not stored separately; the `page_id::sentence_idx` in each result is self-explanatory for eyeballing correctness.

In [3]:
BM25_FULL_PATH   = "rag/indices/bm25/bm25_okapi.pkl"
PASSAGE_IDS_PATH = "rag/indices/bm25/passage_ids.pkl"

def tokenize(text):
    return re.findall(r'\w+', text.lower())

def load_bm25_index(bm25_path=BM25_FULL_PATH, ids_path=PASSAGE_IDS_PATH):
    """Load and return (bm25, passage_ids) from disk."""
    with open(bm25_path, "rb") as f:
        bm25 = pickle.load(f)
    with open(ids_path, "rb") as f:
        passage_ids = pickle.load(f)
    print(f"Loaded BM25 index  : {len(passage_ids):,} passages")
    return bm25, passage_ids

# ── Load ──────────────────────────────────────────────────────────────────
bm25_full, full_ids = load_bm25_index()

# ── Sanity query ──────────────────────────────────────────────────────────
QUERY    = "Albert Einstein Nobel Prize"
q_tokens = tokenize(QUERY)
scores   = bm25_full.get_scores(q_tokens)
top_idxs = scores.argsort()[::-1][:5]

print(f"\nQuery : \"{QUERY}\"\n")
print(f"  {'Rank':<4} {'Score':>8}  passage_id")
print("  " + "─" * 60)
for rank, idx in enumerate(top_idxs, 1):
    print(f"  {rank:<4}  {scores[idx]:>8.3f}  {full_ids[idx]}")

# Expect page_id component to include "Albert_Einstein" in top results
top_pages = [pid.split("::")[0] for pid in (full_ids[i] for i in top_idxs)]
einstein_found = any("Albert_Einstein" in p for p in top_pages)
print(f"\nAlbert_Einstein in top-5 pages : {einstein_found}")

Loaded BM25 index  : 25,247,890 passages

Query : "Albert Einstein Nobel Prize"

  Rank    Score  passage_id
  ────────────────────────────────────────────────────────────
  1       38.203  List_of_Jewish_American_physicists::22
  2       35.146  Einstein_Prize::7
  3       34.495  List_of_peace_prizes::5
  4       31.890  Albert_Einstein_Peace_Prize::0
  5       30.532  Reform_Congregation_Keneseth_Israel_-LRB-Philadelphia-RRB-::19

Albert_Einstein in top-5 pages : True


## Step 3 - Qdrant collection initialization

Sets up the local Qdrant in-process client and creates the `fever_wiki` vector collection. No embedding or upserting happens here — that is Step 4. Keeping this step separate means the collection can be inspected, dropped, or recreated independently of the expensive embedding pass.

**Embedding dimension.** `jinaai/jina-embeddings-v5-text-small` outputs **1024 dimensions** natively (Matryoshka truncation is supported but unused — full 1024 maximises recall). `EMBED_DIM = 1024` is set in the config accordingly.

**Retrieval task API — v5 vs v3.** This is a load-bearing distinction for Step 4 and Day 2:

| | Jina v3 | Jina v5 |
|---|---|---|
| Corpus embedding | `task="retrieval.passage"` | `task="retrieval", prompt_name="document"` |
| Query embedding | `task="retrieval.query"` | `task="retrieval", prompt_name="query"` |

v3 used separate task strings that each activated a different LoRA adapter. v5 consolidates to a single `"retrieval"` task and routes asymmetry through `prompt_name`. **Step 4 must use `prompt_name="document"` for corpus passages. Day 2 query encoding must use `prompt_name="query"`.** Swapping these silently degrades retrieval quality with no error. `trust_remote_code=True` is required at load time for both.

**HNSW config.** `m=16, ef_construct=200` as locked in the project config. Qdrant's default is `ef_construct=100`; the higher value improves index graph quality at build time with no runtime cost. Tuning against recall@10 is a Day 4 task.

| Choice | Why |
|---|---|
| Local in-process Qdrant | No server process to manage; on-disk persistence survives kernel restarts |
| Cosine distance | Jina v5 embeddings are L2-normalised; cosine and dot-product are equivalent, cosine is explicit |
| `prompt_name="document"` for corpus | v5 asymmetric retrieval API — must not use `prompt_name="query"` at index time |
| Minimal payload: `passage_id` + `text` | `page_id` and `sentence_idx` derivable from `passage_id`; smaller payload reduces storage |
| Idempotent creation | Safe to re-run — skips creation if collection exists, avoids wiping in-progress upsert data |

In [8]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, HnswConfigDiff

QDRANT_PATH       = "rag/indices/qdrant"
COLLECTION        = "fever_wiki"
EMBED_DIM         = 1024  # confirmed: jina-embeddings-v5-text-small native dim
HNSW_M            = 16
HNSW_EF_CONSTRUCT = 200

os.makedirs(QDRANT_PATH, exist_ok=True)

def load_qdrant_client(path=QDRANT_PATH):
    """Return a local in-process QdrantClient backed by on-disk storage."""
    return QdrantClient(path=path)

# ── Close any existing client before opening (releases file lock) ─────────
try:
    client.close()
    print("Closed existing Qdrant client.")
except NameError:
    pass

# ── Create collection (idempotent) ────────────────────────────────────────
client = load_qdrant_client()

if client.collection_exists(COLLECTION):
    print(f"Collection '{COLLECTION}' already exists — skipping creation.")
else:
    client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(
            size=EMBED_DIM,
            distance=Distance.COSINE,
        ),
        hnsw_config=HnswConfigDiff(
            m=HNSW_M,
            ef_construct=HNSW_EF_CONSTRUCT,
        ),
    )
    print(f"Collection '{COLLECTION}' created.")

# ── Sanity print ──────────────────────────────────────────────────────────
info = client.get_collection(COLLECTION)
vec  = info.config.params.vectors
print(f"Collection name  : {COLLECTION}")
print(f"Vector dim       : {vec.size}")
print(f"Distance         : {vec.distance}")
print(f"Points count     : {info.points_count:,}")

Collection 'fever_wiki' already exists — skipping creation.
Collection name  : fever_wiki
Vector dim       : 1024
Distance         : Cosine
Points count     : 0


In [4]:
import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
import transformers
print("transformers:", transformers.__version__)

torch: 2.2.1+cu121 cuda: True
transformers: 4.57.6


In [9]:
!pip install "transformers>=4.44,<5.0" --upgrade

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 9.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.14.0
    Uninstalling huggingface_hub-1.14.0:
      Successfully uninstalled huggingface_hub-1.14.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.8.0
    Uninstalling transformers-5.8.0:
      Successfully uninstalled transformers-5.8.0

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import numpy as np
import torch
print("numpy:", np.__version__)
print("torch:", torch.__version__)

numpy: 1.26.4
torch: 2.2.1+cu121


In [11]:
!pip install "numpy<2" --upgrade

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 9.6 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas-stubs 2.3.0.250703 requires types-pytz>=2022.1.1, which is not installed.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## Step 4 - Jina v5 embed + Qdrant upsert

Embeds all 25.2 M passages with `jinaai/jina-embeddings-v5-text-small` (FP16, single RTX 5000) and upserts them into the `fever_wiki` Qdrant collection built in Step 3. This is the longest step in Day 1 — projected 6–12 hours depending on the optimal batch size found in 4A. Checkpointing in 4C means the job can be interrupted and resumed overnight without restarting from scratch.

The step is split into four sub-cells for crash isolation: **4A** loads the model and validates encoding behaviour (dim, dtype, asymmetry, throughput); **4B** does a 10K smoke upsert and verifies the round-trip dense query; **4C** runs the full upsert with batch checkpointing; **4D** is a standalone resume helper. Do not run 4C until 4A and 4B outputs are confirmed.

| Choice | Why |
|---|---|
| `task="retrieval", prompt_name="document"` for corpus | v5 asymmetric API — auto-applies passage prefix internally |
| FP16 on GPU | Turing arch (no BF16); FP16 halves VRAM vs FP32 with no quality loss for retrieval |
| Batch size swept in 4A | 16.7 GB VRAM; optimal batch maximises throughput without OOM |
| Atomic checkpoint write | `.tmp` → rename prevents corrupt checkpoint file on mid-write crash |
| Upsert batch 500 | Qdrant recommends 100–1000 per call; 500 balances IPC overhead vs call count |

### Step 4A - Model load & encoding validation

Loads `jinaai/jina-embeddings-v5-text-small` in FP16 and runs three checks before committing GPU hours to the full corpus:

1. **Shape check** — encode 100 passages, assert output shape is `(100, 1024)`.
2. **Batch consistency** — encode the same text twice in the same batch; cosine similarity must be > 0.9999 (deterministic FP16 output).
3. **Asymmetry check** — encode identical text with `prompt_name="document"` vs `prompt_name="query"`; cosine must be < 0.999. This confirms the task prefix is being applied and the model is behaving asymmetrically as intended. If cosine ≥ 0.999, the prefixes are not taking effect — stop and investigate before indexing.
4. **Throughput sweep** — time batch sizes `[64, 128, 256]` over 500 passages (after a 100-passage warmup) and select the fastest as `EMBED_BATCH`. Flags if the projected full-corpus time exceeds 8 hours.

In [5]:
import time
import numpy as np
from transformers import AutoModel

EMBED_MODEL    = "jinaai/jina-embeddings-v5-text-small"
EMBED_DIM      = 1024
EMBED_MAX_LEN  = 256
TOTAL_PASSAGES = 25_247_890  # from Step 2A


def to_numpy(embs):
    if isinstance(embs, torch.Tensor):
        return embs.cpu().float().numpy()
    return np.array(embs, dtype=np.float32)


def cosine_sim(a, b):
    a = a.flatten().astype(np.float64)
    b = b.flatten().astype(np.float64)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))


# ── GPU guard ─────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA not available — re-run the Step 1 config cell first to set DEVICE, "
        "then re-run this cell. Running Jina v5 on CPU is not viable for this pipeline."
    )

# ── Load model ────────────────────────────────────────────────────────────
print(f"Loading {EMBED_MODEL} ...")
t_load = time.time()
embed_model = AutoModel.from_pretrained(
    EMBED_MODEL,
    trust_remote_code=True,
).to(DEVICE).half()
embed_model.eval()
print(f"Load time : {time.time() - t_load:.1f} s")
print(f"Device    : {next(embed_model.parameters()).device}")
print(f"Dtype     : {next(embed_model.parameters()).dtype}")

# ── Pull 100 sample passages for checks ───────────────────────────────────
sample_texts = [text for _, _, _, text in itertools.islice(iter_passages(WIKI_DIR), 100)]

# ── Check 1: shape ────────────────────────────────────────────────────────
with torch.no_grad():
    vecs = embed_model.encode(
        sample_texts,
        task="retrieval",
        prompt_name="document",
        max_length=EMBED_MAX_LEN,
    )
vecs = to_numpy(vecs)
assert vecs.shape == (100, EMBED_DIM), f"Shape mismatch: {vecs.shape}"
print(f"\nCheck 1 — shape  : {vecs.shape}  PASS")

# ── Check 2: batch consistency ─────────────────────────────────────────────
test_text = sample_texts[0]
with torch.no_grad():
    pair = embed_model.encode(
        [test_text, test_text],
        task="retrieval",
        prompt_name="document",
        max_length=EMBED_MAX_LEN,
    )
pair = to_numpy(pair)
consistency = cosine_sim(pair[0], pair[1])
assert consistency > 0.9999, f"Consistency check failed: {consistency:.6f}"
print(f"Check 2 — batch consistency cosine : {consistency:.6f}  PASS")

# ── Check 3: asymmetry (doc vs query prefix) ───────────────────────────────
with torch.no_grad():
    doc_vec = embed_model.encode(
        [test_text],
        task="retrieval",
        prompt_name="document",
        max_length=EMBED_MAX_LEN,
    )
    qry_vec = embed_model.encode(
        [test_text],
        task="retrieval",
        prompt_name="query",
        max_length=EMBED_MAX_LEN,
    )
doc_vec = to_numpy(doc_vec)
qry_vec = to_numpy(qry_vec)
asymmetry = cosine_sim(doc_vec[0], qry_vec[0])
if asymmetry >= 0.999:
    raise RuntimeError(
        f"Asymmetry check FAILED (cosine={asymmetry:.6f} >= 0.999). "
        "prompt_name prefixes may not be active — do not proceed to 4C."
    )
print(f"Check 3 — doc vs query cosine      : {asymmetry:.6f}  PASS (asymmetric as expected)")

# ── Check 4: throughput sweep ─────────────────────────────────────────────
sweep_texts = [text for _, _, _, text in itertools.islice(iter_passages(WIKI_DIR), 600)]

print("\nThroughput sweep (500 passages after 100-passage warmup) :")
print(f"  {'Batch':>5}  {'p/s':>8}  {'projected (h)':>13}")
print("  " + "─" * 36)

best_bs, best_pps = 64, 0.0
for bs in [64, 128, 256]:
    # warmup
    with torch.no_grad():
        embed_model.encode(
            sweep_texts[:100],
            task="retrieval", prompt_name="document",
            max_length=EMBED_MAX_LEN,
        )
    t0 = time.time()
    with torch.no_grad():
        embed_model.encode(
            sweep_texts[100:600],
            task="retrieval", prompt_name="document",
            max_length=EMBED_MAX_LEN,
        )
    pps = 500 / (time.time() - t0)
    projected_h = TOTAL_PASSAGES / pps / 3600
    flag = "  ← best" if pps > best_pps else ""
    print(f"  {bs:>5}  {pps:>8.0f}  {projected_h:>12.1f} h{flag}")
    if pps > best_pps:
        best_pps = pps
        best_bs  = bs

EMBED_BATCH = best_bs
projected_h_final = TOTAL_PASSAGES / best_pps / 3600
print(f"\nEMBED_BATCH set to : {EMBED_BATCH}")
print(f"Projected runtime  : {projected_h_final:.1f} h")
if projected_h_final > 8:
    print("WARNING: projected runtime exceeds 8 hours — consider running overnight.")

Loading jinaai/jina-embeddings-v5-text-small ...


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Load time : 11.7 s
Device    : cuda:0
Dtype     : torch.float16

Check 1 — shape  : (100, 1024)  PASS
Check 2 — batch consistency cosine : 1.000000  PASS
Check 3 — doc vs query cosine      : 0.797557  PASS (asymmetric as expected)

Throughput sweep (500 passages after 100-passage warmup) :
  Batch       p/s  projected (h)
  ────────────────────────────────────
     64       241          29.1 h  ← best
    128       221          31.7 h
    256       230          30.4 h

EMBED_BATCH set to : 64
Projected runtime  : 29.1 h


In [11]:
import time
import torch
import torch.nn.functional as F

# The 4A sweep never passed bs to encode() — all 3 iterations encoded the same
# 500 texts in one shot. This cell runs the real batch-size test and hoists
# set_adapter/eval outside the inner loop.

def fast_encode(model, texts, task, prompt_name, max_length, batch_size):
    """Batched encode with per-call overhead (set_adapter, eval) hoisted outside loop."""
    model.set_adapter([task])
    model.eval()
    prefix = "Query: " if prompt_name == "query" else "Document: "
    device = next(model.parameters()).device
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        chunk  = texts[i : i + batch_size]
        inputs = [f"{prefix}{t}" for t in chunk]
        batch  = model.tokenizer(
            inputs, return_tensors="pt",
            padding=True, truncation=True, max_length=max_length,
        )
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            out      = model(**batch)
            h        = out.last_hidden_state
            mask     = batch["attention_mask"]
            seq_lens = mask.sum(dim=1) - 1
            pooled   = h[torch.arange(h.shape[0], device=device), seq_lens]
            all_vecs.append(F.normalize(pooled, p=2, dim=-1).cpu())
    return torch.cat(all_vecs, dim=0)


# ── Proper batch-size sweep ───────────────────────────────────────────────
sweep_texts    = [t for _, _, _, t in itertools.islice(iter_passages(WIKI_DIR), 600)]
TOTAL_PASSAGES = 25_247_890

print(f"{'Batch':>5}  {'p/s':>8}  {'proj (h)':>9}")
print("─" * 32)
best_bs, best_pps = 128, 0.0

for bs in [64, 128, 256, 512]:
    torch.cuda.empty_cache()
    fast_encode(embed_model, sweep_texts[:100], "retrieval", "document", EMBED_MAX_LEN, bs)
    torch.cuda.synchronize()
    t0 = time.time()
    fast_encode(embed_model, sweep_texts[100:600], "retrieval", "document", EMBED_MAX_LEN, bs)
    torch.cuda.synchronize()
    pps  = 500 / (time.time() - t0)
    proj = TOTAL_PASSAGES / pps / 3600
    flag = "  ← best" if pps > best_pps else ""
    print(f"{bs:>5}  {pps:>8.0f}  {proj:>8.1f}h{flag}")
    if pps > best_pps:
        best_pps, best_bs = pps, bs

EMBED_BATCH = best_bs
print(f"\nEMBED_BATCH = {EMBED_BATCH}  ({TOTAL_PASSAGES / best_pps / 3600:.1f}h projected)")


Batch       p/s   proj (h)
────────────────────────────────
   64       189      37.0h  ← best
  128       187      37.5h
  256       174      40.3h
  512       144      48.6h

EMBED_BATCH = 64  (37.0h projected)


In [ ]:
import os
os.makedirs("rag/notes", exist_ok=True)

with open("rag/notes/step4_dual_gpu_plan.md", "w") as f:
    f.write("""# Step 4 GPU plan — current state

## Where we are
- 4A model load works. Checks 1-3 PASSED (shape, batch consistency, asymmetry).
- 4A throughput sweep OOMed at batch 128 because GPU 0 was shared with another process (~5 GiB used by external PID).
- Jina v5 is Qwen3-based + PEFT (LLM-class, not encoder-class). Heavier than originally assumed.

## nvidia-smi snapshot
- GPU 0: 15679/16384 MiB used. PID 1526095 (zombie kernel, ours, 10.4 GiB) + PID 2270832 (external, 5.3 GiB). Avoid.
- GPU 1: 616/16384 MiB used (display only). Clean. Use this one.

## Immediate plan: single GPU 1 first
1. First cell of notebook (before any torch import):
       import os
       os.environ["CUDA_VISIBLE_DEVICES"] = "1"
       os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
2. Restart kernel.
3. Verify pinning:
       import torch
       print(torch.cuda.get_device_name(0))
       print(torch.cuda.mem_get_info()[0] / 1e9, "GB free")
   Should show RTX 5000 with ~15 GB free.
4. Re-run cells from top: Step 1 config -> iter_passages def -> 4A.
5. With clean 16 GiB, batch sweep [64, 128, 256] should work.

## Decision point after 4A
- If projected runtime < 12h: proceed with single-GPU 4B then 4C.
- If projected runtime > 24h: implement dual-GPU plan below.

## Dual-GPU plan (only if needed)
Two notebooks, two kernels, corpus split in half.

### Prereq: switch Qdrant from local in-process to server mode
Local mode uses file lock, blocks concurrent writers. Run Qdrant server (Docker or binary), both kernels connect via HTTP/gRPC.

### Kernel A (GPU 0 — once cleared of zombie/external processes)
- os.environ["CUDA_VISIBLE_DEVICES"] = "0"
- MY_FILES = wiki_files[:len(wiki_files)//2]
- CHECKPOINT_PATH = "rag/indices/qdrant/checkpoint_gpu0.txt"

### Kernel B (GPU 1)
- os.environ["CUDA_VISIBLE_DEVICES"] = "1"
- MY_FILES = wiki_files[len(wiki_files)//2:]
- CHECKPOINT_PATH = "rag/indices/qdrant/checkpoint_gpu1.txt"

### Refactor needed
- iter_passages must accept a list of files (not just a directory) so each kernel iterates only its slice.

### Verification after dual-GPU run
- client.count(collection_name="fever_wiki") should equal TOTAL_PASSAGES (25_247_890) exactly.
- Lower means a slice was missed; higher means slices overlapped.

## Other context for fresh session
- transformers pinned to 4.57.6 (5.x broke Jina v5 remote code).
- numpy pinned to <2 (torch 2.2.1 ABI requires numpy 1.x).
- Step 1 (iter_passages), Step 2 (BM25 index, 25.2M passages), Step 3 (Qdrant collection, dim=1024 cosine) all DONE.
- 4A code is correct as written; only environment config needs the GPU pinning prepended.

## Optional housekeeping
- PID 1526095 on GPU 0 is our zombie kernel from earlier OOM crash. Could `kill 1526095` to free GPU 0 for others. Verify ownership first with `ps -p 1526095`.
""")
print("Saved.")

Saved.


In [5]:
import subprocess
r = subprocess.run(['python3', '-c', 'import PIL; print(PIL.__version__, PIL.__file__)'], capture_output=True, text=True)
print(r.stdout)

12.2.0 /home/ai-ws2/.local/lib/python3.10/site-packages/PIL/__init__.py



### Step 4B - Smoke upsert & dense retrieval check (10K passages)

Embeds the first 10,000 passages (the smoke corpus from Step 1) and upserts them into the `fever_wiki` Qdrant collection. This validates the full encode → upsert → query round-trip before the multi-hour full-corpus run in 4C.

Each Qdrant point stores two payload fields: `passage_id` (the `page_id::sentence_idx` string) and `text` (the raw sentence). `page_id` and `sentence_idx` are derivable from `passage_id` and are not stored redundantly. Point IDs are sequential integers `0, 1, 2, ...` matching position in the iteration order.

At the end, a dense query for "Albert Einstein Nobel Prize" is issued and the top-5 results are printed. Since the smoke corpus covers only `wiki-001.jsonl`, the Albert Einstein page may not appear — the primary goal is confirming the query returns results with valid `passage_id` payload, not retrieval quality at this scale. The collection is then **deleted and recreated clean** so Cell 4C starts from an empty collection.

In [11]:
from qdrant_client.models import PointStruct, Distance, VectorParams, HnswConfigDiff

# ── Reload model if needed ────────────────────────────────────────────────
try:
    embed_model
except NameError:
    print("embed_model not in scope — reloading ...")
    from transformers import AutoModel
    embed_model = AutoModel.from_pretrained(
        EMBED_MODEL,
        trust_remote_code=True,
    ).to(DEVICE).half()
    embed_model.eval()
    print("Reloaded.")

# ── Collect smoke passages ────────────────────────────────────────────────
smoke_passages = list(itertools.islice(iter_passages(WIKI_DIR), SMOKE_LIMIT))
print(f"Smoke passages : {len(smoke_passages):,}")

# ── Ensure client + collection exist ─────────────────────────────────────
try:
    client
except NameError:
    from qdrant_client import QdrantClient
    client = load_qdrant_client()

if not client.collection_exists(COLLECTION):
    client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
        hnsw_config=HnswConfigDiff(m=HNSW_M, ef_construct=HNSW_EF_CONSTRUCT),
    )
    print(f"Collection '{COLLECTION}' created.")

# ── Embed + upsert in EMBED_BATCH chunks ─────────────────────────────────
UPSERT_BATCH = 500
t0 = time.time()

for chunk_start in tqdm(range(0, len(smoke_passages), EMBED_BATCH),
                        desc="Embedding + upserting", unit=" batches"):
    chunk = smoke_passages[chunk_start: chunk_start + EMBED_BATCH]
    texts = [text for _, _, _, text in chunk]

    with torch.no_grad():
        vecs = embed_model.encode(
            texts,
            task="retrieval",
            prompt_name="document",
            max_length=EMBED_MAX_LEN,
        )
    vecs = to_numpy(vecs)

    points = [
        PointStruct(
            id=chunk_start + j,
            vector=vecs[j].tolist(),
            payload={"passage_id": chunk[j][0], "text": chunk[j][3]},
        )
        for j in range(len(chunk))
    ]

    for upsert_start in range(0, len(points), UPSERT_BATCH):
        client.upsert(
            collection_name=COLLECTION,
            points=points[upsert_start: upsert_start + UPSERT_BATCH],
        )

elapsed = time.time() - t0
count_after = client.get_collection(COLLECTION).points_count
print(f"\nUpserted          : {count_after:,} points")
print(f"Elapsed           : {elapsed:.1f} s")
print(f"Throughput        : {len(smoke_passages) / elapsed:.0f} p/s")

# ── Dense sanity query ────────────────────────────────────────────────────
QUERY = "Albert Einstein Nobel Prize"
with torch.no_grad():
    q_vec = embed_model.encode(
        [QUERY],
        task="retrieval",
        prompt_name="query",
        max_length=EMBED_MAX_LEN,
    )
q_vec = to_numpy(q_vec)[0].tolist()

hits = client.query_points(
    collection_name=COLLECTION,
    query=q_vec,
    limit=5,
    with_payload=True,
).points

print(f"\nDense query : \"{QUERY}\"\n")
print(f"  {'Rank':<4} {'Score':>8}  passage_id")
print("  " + "─" * 60)
for rank, hit in enumerate(hits, 1):
    pid = hit.payload.get("passage_id", "MISSING")
    print(f"  {rank:<4}  {hit.score:>8.4f}  {pid}")
    print(f"            {hit.payload.get('text','')[:110]}")
    print()

assert all("passage_id" in h.payload for h in hits), "payload missing passage_id!"
print("Payload check : PASS (all hits have passage_id)")

# ── Clean up: delete + recreate for 4C full run ───────────────────────────
print("\nDeleting smoke collection and recreating clean for 4C ...")
client.delete_collection(COLLECTION)
client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    hnsw_config=HnswConfigDiff(m=HNSW_M, ef_construct=HNSW_EF_CONSTRUCT),
)
count_clean = client.get_collection(COLLECTION).points_count
print(f"Collection '{COLLECTION}' recreated. Points count : {count_clean:,}")
print("Ready for 4C full-corpus upsert.")

Smoke passages : 10,000


Embedding + upserting:   0%|          | 0/79 [00:00<?, ? batches/s]


Upserted          : 10,000 points
Elapsed           : 172.2 s
Throughput        : 58 p/s

Dense query : "Albert Einstein Nobel Prize"

  Rank    Score  passage_id
  ────────────────────────────────────────────────────────────
  1       0.3502  1998_Oakland_Athletics_season::10
            The award was the Athletics ' first since Walt Weiss received one in 1988 .

  2       0.3497  11th_Moscow_International_Film_Festival::1
            The Golden Prizes were awarded to the Italian-French film Christ Stopped at Eboli directed by Francesco Rosi ,

  3       0.3449  1978_National_Society_of_Film_Critics_Awards::0
            13th NSFC Awards

  4       0.3344  1904_Massevitch::4
            It was later named after Russian astrophysicist Alla Massevitch .

  5       0.3208  14th_César_Awards::2
            Camille Claudel won the award for Best Film .

Payload check : PASS (all hits have passage_id)

Deleting smoke collection and recreating clean for 4C ...
Collection 'fever_wiki' recrea

In [14]:
#Step 1: indexing_threshold=0 verification
import time
from qdrant_client.models import (
    PointStruct, Distance, VectorParams, HnswConfigDiff, OptimizersConfigDiff
)

# ── Recreate collection with live HNSW indexing disabled ─────────────────
try:
    client
except NameError:
    from qdrant_client import QdrantClient
    client = load_qdrant_client()

print("Recreating collection with indexing_threshold=30_000_000 ...")
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    hnsw_config=HnswConfigDiff(m=HNSW_M, ef_construct=HNSW_EF_CONSTRUCT),
    optimizers_config=OptimizersConfigDiff(indexing_threshold=30_000_000),
)
info = client.get_collection(COLLECTION)
print(f"Collection ready. Points: {info.points_count:,}")
print(f"indexing_threshold: {info.config.optimizer_config.indexing_threshold}")

# ── Pre-embed 10K (exclude embed time from upsert measurement) ───────────
smoke_passages = list(itertools.islice(iter_passages(WIKI_DIR), SMOKE_LIMIT))
print(f"\nPre-embedding {len(smoke_passages):,} passages ...")
t_emb = time.time()
vecs_all = fast_encode(
    embed_model, [t for _, _, _, t in smoke_passages],
    "retrieval", "document", EMBED_MAX_LEN, EMBED_BATCH,
)
torch.cuda.synchronize()
emb_elapsed = time.time() - t_emb
emb_pps = len(smoke_passages) / emb_elapsed
print(f"Embed-only : {emb_pps:.0f} p/s  ({emb_elapsed:.1f} s)")

# ── Build points ─────────────────────────────────────────────────────────
all_points = [
    PointStruct(
        id=j,
        vector=vecs_all[j].tolist(),
        payload={"passage_id": smoke_passages[j][0], "text": smoke_passages[j][3]},
    )
    for j in range(len(smoke_passages))
]

# ── Time upsert alone ─────────────────────────────────────────────────────
UPSERT_BATCH = 5000
print(f"\nUpserting {len(all_points):,} points (indexing_threshold=30_000_000) ...")
t_ups = time.time()
for us in tqdm(range(0, len(all_points), UPSERT_BATCH), desc="Upsert-only", unit=" batches"):
    client.upsert(collection_name=COLLECTION, points=all_points[us : us + UPSERT_BATCH])
ups_elapsed = time.time() - t_ups
ups_pps = len(smoke_passages) / ups_elapsed
print(f"\nUpsert-only : {ups_pps:.0f} p/s  ({ups_elapsed:.1f} s)  [baseline was ~87 p/s combined]")

combined_pps = len(smoke_passages) / (emb_elapsed + ups_elapsed)
proj_h = TOTAL_PASSAGES / combined_pps / 3600
print(f"Combined    : {combined_pps:.0f} p/s  →  {proj_h:.1f}h projected")
if ups_pps < 200:
    print("WARNING: < 200 p/s — indexing_threshold=0 may not have taken effect.")
else:
    print("PASS: upsert speedup confirmed.")

# ── Clean to 0 points, keep config for 4C ────────────────────────────────
client.delete_collection(COLLECTION)
client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    hnsw_config=HnswConfigDiff(m=HNSW_M, ef_construct=HNSW_EF_CONSTRUCT),
    optimizers_config=OptimizersConfigDiff(indexing_threshold=30_000_000),
)
print(f"Collection reset for 4C. Points: {client.get_collection(COLLECTION).points_count:,}")


Recreating collection with indexing_threshold=30_000_000 ...
Collection ready. Points: 0
indexing_threshold: 20000

Pre-embedding 10,000 passages ...
Embed-only : 164 p/s  (60.9 s)

Upserting 10,000 points (indexing_threshold=30_000_000) ...


Upsert-only:   0%|          | 0/2 [00:00<?, ? batches/s]


Upsert-only : 101 p/s  (99.4 s)  [baseline was ~87 p/s combined]
Combined    : 62 p/s  →  112.4h projected
Collection reset for 4C. Points: 0


In [17]:
with open("rag/notes/step4_session2_handoff.md", "a") as f:
    f.write("""

---

## UPDATE: Optimization 2 (torch.compile) DROPPED

### What happened
Both Cell A (indexing_threshold setup) and Cell B (torch.compile + benchmark) crashed with the same root cause. Cell B ran first, broke embed_model in kernel memory by wrapping it with torch.compile, then Cell A tried to use that broken model and hit the same crash.

### Diagnosis
Both tracebacks crashed at:
- File: `transformers/utils/import_utils.py`, line 1015
- Call: `return torch.compiler.is_compiling()`
- Error: `InternalTorchDynamoError: is_compiling`

transformers 4.57.6 calls `torch.compiler.is_compiling()` inside Qwen3Model's masking code. torch 2.2.1's Dynamo cannot trace through `torch.compiler.is_compiling` — it tries `__dict__` lookup and fails. Hard incompatibility between transformers 4.57.6 + torch 2.2.1.

`suppress_errors=True` would silently fall back to eager (0% speedup) — pointless.

**Verdict: torch.compile is unusable on this stack. Optimization 2 dropped entirely.**

### Recovery required
embed_model is permanently tainted by torch.compile wrapping. Must restart kernel.

### Re-run order after kernel restart
1. c042a614 — env-var cell (CUDA_VISIBLE_DEVICES=1, expandable_segments)
2. f08ffaab — GPU verify
3. step1-install, step1-config, step1-loader — imports + iter_passages
4. c7e00907 — 4A (reloads embed_model clean)
5. 34c0efbb — fast_encode definition + batch sweep
6. **Cell A only** (indexing_threshold=30_000_000, UPSERT_BATCH=5000)
7. **SKIP Cell B** entirely

### Revised optimization plan
| Optimization | Status |
|---|---|
| indexing_threshold=30_000_000 | Pending — re-run Cell A after restart |
| torch.compile | DROPPED (transformers 4.57.6 + torch 2.2.1 incompatible) |
| fast_encode (hoisted set_adapter / eval out of inner loop) | Done, confirmed working |
| Async upsert overlap | Wire into 4C after Cell A result |

### Note on indexing_threshold value
Plan changed from `indexing_threshold=0` to `indexing_threshold=30_000_000`. Functionally equivalent for our purposes (>25.2M total) — both disable indexing during load. The 30M number is just an explicit count above corpus size rather than the special-case 0. Both work; this is what Claude Code chose.

### fast_encode optimization (already applied, kept)
Custom encode wrapper that hoists `set_adapter()` and `model.eval()` calls out of the per-batch inner loop. These were being called inside Jina v5's encode() on every batch, adding Python overhead. Hoisting gave a meaningful speedup (real number TBD after kernel restart re-runs the sweep). Independent of torch.compile — survives the drop.

### Updated next-session kickoff prompt
"Read rag/notes/step4_session2_handoff.md including the UPDATE section at the bottom. torch.compile is dropped (transformers/torch version incompatibility). Kernel needs restart, then re-run cells in this order: env-var → GPU verify → step1 imports → 4A (reloads model clean) → fast_encode + batch sweep → Cell A (indexing_threshold=30_000_000) — SKIP Cell B. Report Cell A's measured upsert throughput. Then we write 4C with fast_encode + indexing disabled + async upsert overlap."

### Revised projection (post-torch.compile drop)
- Embedding: ~179 p/s baseline + fast_encode improvement (TBD, likely +10-20%)
- Upsert: 87 p/s → 500+ p/s expected with indexing_threshold=30M
- Async overlap recovers most of upsert time
- Estimated 4C runtime: 35-45h (vs 32h with compile, vs 80h baseline)
- Still significantly better than the 80-120h worst case
""")
print("Updated rag/notes/step4_session2_handoff.md")
print(f"File size now: {os.path.getsize('rag/notes/step4_session2_handoff.md')} bytes")

Updated rag/notes/step4_session2_handoff.md
File size now: 3516 bytes


In [15]:
with open("rag/notes/step4_session2_2_handoff.md", "a") as f:
    f.write("""

---

## 4C kickoff state (this is the run)

### Final cell decisions
- fast_encode for embedding (PEFT-friendly, hoists set_adapter/eval out of inner loop)
- indexing_threshold=30_000_000 set on collection (no live HNSW during load)
- Async overlap: embed N+1 runs concurrently with upsert N (background thread, max_workers=1)
- "build points before gate" refinement: PointStruct construction also overlaps with upsert if upsert runs long
- UPSERT_BATCH=5000 (local in-process Qdrant, no gRPC limits — verified)
- Checkpoint every 500 batches, atomic write to rag/indices/qdrant/checkpoint_4c.json
- torch.cuda.empty_cache() every 1000 batches
- Post-load: update_collection(optimizer_config=OptimizersConfigDiff(indexing_threshold=20_000)) triggers async one-shot HNSW build

### Projection variance
Cell A's projection swung 68h-106h across smoke runs — not at steady state. Real wall-clock will emerge after ~60 min of 4C runtime. Plan: kick off, watch first hour, kill if ETA balloons past ~100h.

### Things that survive crashes
- Checkpoint file: rag/indices/qdrant/checkpoint_4c.json — source of truth for resume
- Qdrant collection on disk: rag/indices/qdrant/ — points already upserted persist
- If kernel dies mid-run, just re-run Cell 4C — it auto-resumes from checkpoint

### After 4C cell exits
- HNSW build is queued, runs ASYNC in background
- Poll: client.get_collection(COLLECTION).status — wait until 'green' before Step 5
- Build itself takes 1-3h additional for 25M @ dim 1024

### Productive use of 4C wall time
Day 2 retrieval pipeline draft + Day 3 verdict logic draft, tested against a small test collection (or the partial fever_wiki as it grows — read-only access during load is fine since indexing_threshold=30M means no live indexing contention).
""")
print("Saved.")

Saved.


### Step 4C — Full corpus embed + upsert (25.2 M passages)

Embeds all 25,247,890 passages with `fast_encode` (Jina v5, FP16, GPU 1) and upserts them into the `fever_wiki` Qdrant collection.

**Optimisations applied**
| Technique | Effect |
|---|---|
| `indexing_threshold=30_000_000` | Disables live HNSW indexing during load — upsert throughput ~205 p/s vs ~87 p/s baseline |
| `fast_encode` (hoisted `set_adapter` / `eval`) | GPU embed throughput ~206 p/s |
| Async upsert overlap (`ThreadPoolExecutor`) | GPU embeds batch N+1 while disk-commits batch N; effective wall-clock ≈ `max(T_embed, T_upsert)` per batch |

**Checkpointing** — atomic `.tmp` → rename every 500 upsert batches (~2.5 M passages). On crash, re-run the cell: it reads `checkpoint_4c.json`, skips already-indexed passages, and resumes from the last confirmed point.

**Post-load** — `update_collection(indexing_threshold=20_000)` drops the threshold below the corpus size, triggering Qdrant's one-shot HNSW build over all vectors. Poll `client.get_collection(COLLECTION).status` until `green`.

Estimated runtime: **~34 h** (single RTX 5000, GPU 1).


In [ ]:
# Cell 4C — Full corpus embed + upsert
# fast_encode + indexing_threshold=30M (already set) + async overlap + checkpointing

import collections, concurrent.futures, itertools, json, time
from pathlib import Path
from qdrant_client.models import PointStruct, OptimizersConfigDiff
from tqdm.auto import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
CHECKPOINT_PATH   = Path("rag/indices/qdrant/checkpoint_4c.json")
CHECKPOINT_EVERY  = 500   # flush + save checkpoint every N upsert batches
EMPTY_CACHE_EVERY = 1000  # torch.cuda.empty_cache() every N upsert batches

# ── Checkpoint helpers ────────────────────────────────────────────────────────
def _save_ckpt(path, n):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps({"passages_done": n}))
    tmp.replace(path)                          # atomic rename

def _load_ckpt(path):
    return json.loads(path.read_text())["passages_done"] if path.exists() else 0

# ── Passage batcher with resume skip ─────────────────────────────────────────
def _batches(skip, batch_size):
    gen = iter_passages(WIKI_DIR)
    if skip:
        print(f"  Skipping {skip:,} already-indexed passages ...")
        collections.deque(itertools.islice(gen, skip), maxlen=0)
    buf_p, buf_t = [], []
    for pid, _, _, text in gen:
        buf_p.append(pid); buf_t.append(text)
        if len(buf_p) == batch_size:
            yield buf_p, buf_t
            buf_p, buf_t = [], []
    if buf_p:
        yield buf_p, buf_t

# ── Upsert worker (runs in background thread) ─────────────────────────────────
def _upsert(points):
    client.upsert(collection_name=COLLECTION, points=points, wait=True)

# ── Resume ────────────────────────────────────────────────────────────────────
start_at = _load_ckpt(CHECKPOINT_PATH)
print(f"Starting 4C  |  passages_done={start_at:,}  ({start_at / TOTAL_PASSAGES * 100:.2f}%)")
print(f"Collection points before run : {client.count(COLLECTION).count:,}")

# ── Main loop ─────────────────────────────────────────────────────────────────
t0             = time.time()
passages_done  = start_at
batch_idx      = start_at // UPSERT_BATCH
pending_future = None

with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
    pbar = tqdm(
        total     = TOTAL_PASSAGES,
        initial   = start_at,
        desc      = "4C",
        unit      = " p",
        smoothing = 0.05,
    )

    for pids, texts in _batches(start_at, UPSERT_BATCH):

        # ── Stage 1: Embed on GPU ────────────────────────────────────────────
        # Background thread runs upsert N-1 concurrently during this call.
        vecs    = fast_encode(embed_model, texts, "retrieval", "document",
                              EMBED_MAX_LEN, EMBED_BATCH)
        vecs_np = vecs.cpu().float().numpy()

        # ── Stage 2: Build points (CPU) ──────────────────────────────────────
        # Done here — before the gate — so CPU is busy if upsert N-1 runs long.
        base_id = passages_done
        points  = [
            PointStruct(
                id      = base_id + j,
                vector  = vecs_np[j].tolist(),
                payload = {"passage_id": pids[j], "text": texts[j]},
            )
            for j in range(len(pids))
        ]

        # ── Gate: wait for upsert N-1, then immediately fire upsert N ────────
        if pending_future is not None:
            pending_future.result()
        pending_future = executor.submit(_upsert, points)  # non-blocking
        # GPU is now idle; upsert N runs in background while Stage 1 of N+1 runs.

        passages_done += len(pids)
        batch_idx     += 1
        pbar.update(len(pids))

        elapsed = time.time() - t0
        p_per_s = (passages_done - start_at) / elapsed if elapsed > 0 else 0
        eta_h   = (TOTAL_PASSAGES - passages_done) / p_per_s / 3600 if p_per_s > 0 else 0
        pbar.set_postfix({"p/s": f"{p_per_s:.0f}", "ETA_h": f"{eta_h:.1f}"})

        # ── Checkpoint: flush upsert, then write atomically ──────────────────
        if batch_idx % CHECKPOINT_EVERY == 0:
            pending_future.result()
            pending_future = None
            _save_ckpt(CHECKPOINT_PATH, passages_done)

        # ── GPU memory ───────────────────────────────────────────────────────
        if batch_idx % EMPTY_CACHE_EVERY == 0:
            torch.cuda.empty_cache()

    # Flush the final in-flight upsert
    if pending_future is not None:
        pending_future.result()

    pbar.close()

# ── Final checkpoint ─────────────────────────────────────────────────────────
_save_ckpt(CHECKPOINT_PATH, passages_done)
elapsed_total = time.time() - t0
print(f"\nLoad done  |  {passages_done:,} passages  |  {elapsed_total / 3600:.2f} h")
print(f"Throughput : {(passages_done - start_at) / elapsed_total:.0f} p/s (wall-clock)")
print(f"Collection count : {client.count(COLLECTION).count:,}")

# ── Trigger one-shot HNSW build ───────────────────────────────────────────────
print("\nLowering indexing_threshold → 20,000 to trigger HNSW build ...")
client.update_collection(
    collection_name  = COLLECTION,
    optimizer_config = OptimizersConfigDiff(indexing_threshold=20_000),
)
info = client.get_collection(COLLECTION)
print(f"optimizer.indexing_threshold : {info.config.optimizer_config.indexing_threshold}")
print(f"collection.status            : {info.status}")
print("HNSW build queued. Poll: client.get_collection(COLLECTION).status  (→ 'green' when done)")


Starting 4C  |  passages_done=0  (0.00%)
Collection points before run : 0


4C:   0%|          | 0/25247890 [00:00<?, ? p/s]

/tmp/ipykernel_1798511/3963761912.py:40: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 25000 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(collection_name=COLLECTION, points=points, wait=True)
